In [1]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
import ast

In [2]:
def rpy_to_rotmat(rpy):
    """
    rpy: (..., 3) roll, pitch, yaw
    returns: (..., 3, 3) rotation matrix (base -> world)
    """
    roll, pitch, yaw = rpy[..., 0], rpy[..., 1], rpy[..., 2]

    cr, sr = np.cos(roll),  np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw),   np.sin(yaw)

    R = np.stack([
        np.stack([cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr], axis=-1),
        np.stack([sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr], axis=-1),
        np.stack([-sp,   cp*sr,            cp*cr           ], axis=-1)
    ], axis=-2)

    return R


In [3]:
# Define a function to convert the string representation of an array/list
def string_to_array(array_string):
    try:
        # Use ast.literal_eval to safely evaluate the string as a Python literal
        return ast.literal_eval(array_string)
    except (ValueError, SyntaxError):
        # Handle cases where the string might not be a valid list/array string
        return array_string # Or some default/error value

In [4]:
exp_filepath = "exp_data/scratch_pact_exp/plane_tracking_test.csv"

df = pd.read_csv(exp_filepath, converters={'base_cmd': string_to_array,
                                           'base_pose': string_to_array,
                                           'base_rpy': string_to_array,
                                           'q_actual': string_to_array,
                                           'base_lin_vel': string_to_array,
                                           'base_ang_vel': string_to_array,
                                           'dof_vel': string_to_array,
                                           'proj_grav': string_to_array,
                                           'feet_pos': string_to_array,
                                           'tau_act': string_to_array,
                                           'grf': string_to_array,
                                           'q_des': string_to_array,
                                           'tau_ff': string_to_array,
                                           'tau_pd': string_to_array,
                                           'failure': string_to_array})

print(len(df))

KeyboardInterrupt: 

In [ ]:
# base_cmd	base_pose	base_rpy	q_actual	base_lin_vel	base_ang_vel	dof_vel	proj_grav	feet_pos


# Joint tracking / violation metrics
q_observations = np.array(df["q_actual"].to_list())

# cmd tracking metrics
vel_cmds = np.array(df["base_cmd"].to_list())  # v_x, v_y, w_yaw
lin_vel  = np.array(df["base_lin_vel"].to_list())
ang_vel  = np.array(df["base_ang_vel"].to_list())   # stability via roll/pitch angular velo.

# Oreintation
proj_grav = np.array(df["proj_grav"].to_list())

# Height tracking metric
base_pose = np.array(df["base_pose"].to_list())
base_rpy = np.array(df["base_rpy"].to_list())

R_w_b = rpy_to_rotmat(base_rpy)

joint_limits = np.array([[-1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721],
                         [1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837]])

joint_torque_limits = np.array([23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55])

In [ ]:
lin_cmd_errors = vel_cmds[:,0:2] - lin_vel[:,0:2]
ang_cmd_errors = vel_cmds[:,2] - ang_vel[:,2]

cmd_errs = np.concatenate((lin_cmd_errors, ang_cmd_errors[:,None]), axis=1)

print(cmd_errs.shape)

print("Linear CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(lin_cmd_errors))),4))
print("Linear CMD Tracking STDDV: ", np.round(np.sqrt(np.std(np.square(lin_cmd_errors))),4))
print("Linear CMD Tracking  MAE: ", np.round(np.mean(np.abs(lin_cmd_errors)),4))
print("Linear CMD Tracking  MAE: ", np.round(np.median(np.abs(lin_cmd_errors)),4))


print("Angular CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(ang_cmd_errors))),4))
print("Angular CMD Tracking STDDV: ", np.round(np.sqrt(np.std(np.square(ang_cmd_errors))),4))
print("Angular CMD Tracking  MAE: ", np.round(np.mean(np.abs(ang_cmd_errors)),4))
print("Angular CMD Tracking  MAE: ", np.round(np.median(np.abs(ang_cmd_errors)),4))


print("Total CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(cmd_errs))),4))
print("Total CMD Tracking  STDDV: ", np.round(np.sqrt(np.std(np.square(cmd_errs))),4))

In [ ]:
height_errors = 0.32 - base_pose[:,2]

print("Height CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(height_errors))),4))
print("Height CMD Tracking  MAE: ", np.round(np.mean(np.abs(height_errors)),4))

In [ ]:
orientation_norm = np.linalg.norm(proj_grav[:,0:2], axis=1)

print("Projected Grav. R/P Norm: ", np.round(np.mean(orientation_norm),4))

In [ ]:
velo_z = lin_vel[:,2]
ang_velo_norm = np.linalg.norm(ang_vel[:,0:2], axis=1)

print("Angular Velo. R/P Norm MEAN: ", np.round(np.mean(ang_velo_norm),4))
print("Angular Velo. R/P Norm STD: ", np.round(np.std(ang_velo_norm),4))

print("Z Velo. MEAN: ", np.round(np.sqrt(np.mean(np.square(velo_z))),4))
print("Z Velo. STD: ", np.round(np.sqrt(np.std(np.square(velo_z))),4))

total_unwatnted_velo = np.concatenate((velo_z[:,None], ang_vel[:,0:2]), axis=1)

print("Total Velo. R/P Norm MEAN: ", np.round(np.mean(np.linalg.norm(total_unwatnted_velo, axis=1)),4))
print("Angular Velo. R/P Norm STD: ", np.round(np.std(np.linalg.norm(total_unwatnted_velo, axis=1)),4))

In [ ]:
joint_pred_limits_error = -(q_observations - joint_limits[0,:]).clip(max=0.0)
joint_pred_limits_error += (q_observations - joint_limits[1,:]).clip(min=0.0)

print("Avg. Joint Violation: ", np.mean(joint_pred_limits_error))